# Perform NicheNet Analysis Starting from an AnnData Object (Wrapper)

In this notebook, you can learn how to perform a basic NicheNet analysis on an AnnData object containing single-cell expression data using the wrapper function `nichenet_seuratobj_aggregate`. This wrapper performs the same steps described in the step-by-step analysis notebook, but in a single function call.

As example expression data, we use mouse NICHE-seq data exploring intercellular communication in the T cell area in the inguinal lymph node before and 72 hours after LCMV infection (Medaglia et al., 2017). We will focus on CD8 T cells as the receiver population.

## Prepare NicheNet Analysis

### Load packages

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import nichenetr as nn

### Read in NicheNet's networks

In [ ]:
organism = "mouse"

lr_network = nn.load_lr_network(organism)
ligand_target_matrix = nn.load_ligand_target_matrix(organism)
weighted_networks = nn.load_weighted_networks(organism)

lr_network = lr_network[["from", "to"]].drop_duplicates()
print(lr_network.head())
print(f"\nLigand-target matrix: {ligand_target_matrix.data.shape}")
print(weighted_networks["lr_sig"].head())
print(weighted_networks["gr"].head())

### Read in the expression data

In [ ]:
adata = nn.load_seurat_obj()
adata = nn.alias_to_symbol_anndata(adata, "mouse")

adata.obs.head()

In [ ]:
print(adata.obs["celltype"].value_counts())
sc.pl.tsne(adata, color="celltype")

In [ ]:
print(adata.obs["aggregate"].value_counts())
sc.pl.tsne(adata, color="aggregate")

## Perform the NicheNet Analysis

### Sender-agnostic approach

For the sender-agnostic approach, the sender is set to `"undefined"`. The receiver is CD8 T cells, and the gene set of interest is the set of DE genes after LCMV infection.

In [ ]:
nichenet_output_agnostic = nn.nichenet_seuratobj_aggregate(
    receiver="CD8 T",
    adata=adata,
    sender="undefined",
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    celltype_col="celltype",
    expression_pct=0.05,
    ligand_target_matrix=ligand_target_matrix,
    lr_network=lr_network,
    weighted_networks=weighted_networks,
)

### Sender-focused approach

Provide one or more sender cell populations:

In [ ]:
nichenet_output = nn.nichenet_seuratobj_aggregate(
    receiver="CD8 T",
    adata=adata,
    sender=["CD4 T", "Treg", "Mono", "NK", "B", "DC"],
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    celltype_col="celltype",
    expression_pct=0.05,
    ligand_target_matrix=ligand_target_matrix,
    lr_network=lr_network,
    weighted_networks=weighted_networks,
)

### Interpret the NicheNet analysis output

We investigate the output of the sender-focused approach.

In [ ]:
print("Output keys:", list(nichenet_output.keys()))

#### Ligand activity analysis results

In [ ]:
nichenet_output["ligand_activities"]

In [ ]:
# Top 30 ligands
print(nichenet_output["top_ligands"])

In [ ]:
# Ligand-target heatmap
nichenet_output["ligand_target_heatmap"]
plt.show()

In [ ]:
# Ligand-target links as a DataFrame
nichenet_output["ligand_target_df"].head(10)

In [ ]:
# Top targets
print(nichenet_output["top_targets"][:20])

#### Inferred ligand-receptor interactions

In [ ]:
# Ligand-receptor heatmap
nichenet_output["ligand_receptor_heatmap"]
plt.show()

In [ ]:
# Ligand-receptor links as a DataFrame
nichenet_output["ligand_receptor_df"].head(10)

In [ ]:
# Top receptors
print(nichenet_output["top_receptors"])

In [ ]:
# Gene set and background genes
print(f"Gene set size: {len(nichenet_output['geneset_oi'])}")
print(f"Background genes: {len(nichenet_output['background_expressed_genes'])}")

### Results of the sender-agnostic approach

In [ ]:
nichenet_output_agnostic["ligand_target_heatmap"]
plt.show()

### Running multiple NicheNet analyses on different receiver cell populations

You can run the analysis for every receiver cell type:

In [ ]:
receiver_celltypes_oi = ["CD4 T", "CD8 T"]

nichenet_outputs = {}
for receiver_ct in receiver_celltypes_oi:
    nichenet_outputs[receiver_ct] = nn.nichenet_seuratobj_aggregate(
        receiver=receiver_ct,
        adata=adata,
        condition_col="aggregate",
        condition_oi="LCMV",
        condition_ref="SS",
        sender=["CD4 T", "Treg", "Mono", "NK", "B", "DC"],
        celltype_col="celltype",
        ligand_target_matrix=ligand_target_matrix,
        lr_network=lr_network,
        weighted_networks=weighted_networks,
    )

In [ ]:
# Check common and cell-type-specific ligands
cd4_ligands_set = set(nichenet_outputs["CD4 T"]["top_ligands"])
cd8_ligands_set = set(nichenet_outputs["CD8 T"]["top_ligands"])

common_ligands = cd4_ligands_set & cd8_ligands_set
print("Common ligands:", sorted(common_ligands))
print("\nCD4 T-specific:", sorted(cd4_ligands_set - cd8_ligands_set))
print("\nCD8 T-specific:", sorted(cd8_ligands_set - cd4_ligands_set))